# Part 1b - Downscale to target 1.3 km & build the submission

In [ ]:
import os, sys, warnings
from pathlib import Path
try:
    _here = Path(__vsc_ipynb_file__).resolve().parent   # VS Code sets this
except NameError:
    _here = Path.cwd().resolve()
_root = next(d for d in [_here, *_here.parents]
             if (d / 'part0_dataset_setup' / 'target_loader.py').exists())
os.chdir(_root)
sys.path[:0] = ['.', 'part0_dataset_setup', 'part1_forecast',
                'part2_siting', 'part3_economics']
warnings.filterwarnings('ignore')
import config; print(config.describe())
import pickle, numpy as np, pandas as pd
import forecast_pipeline as P
import build_forecast_submission as bfs
import downscaling as dn, target_loader, splits
with open('part1_forecast/cache/coarse_forecasts.pkl', 'rb') as f:
    cache = pickle.load(f)
offs = cache['offs']

## 1. Train the terrain downscaler

In [ ]:
d2020 = [d for d in target_loader.list_dates(config.target_root()) if d.year == 2020][::5]
dwn = dn.train_downscaler(d2020, hours=(0, 6, 12, 18))
print('downscaler trained on', len(d2020), 'days')
mos, qmos, adj = cache['models']
spd_infl, dir_off = P.calibrate_intervals(mos, qmos, adj, dwn, offs)
print('speed widening per lead:', spd_infl, '| dir half-width:', dir_off)

## 2. Downscale every (window, lead, hour) and assemble the submission

In [ ]:
windows = splits.eval_windows()
blocks = []
for wi in range(len(windows)):
    blocks += P.downscale_window(dwn, cache[wi], offs, wi, spd_infl=spd_infl, dir_off=dir_off)
    print(f'window {wi}: downscaled')
sub = bfs.assemble(blocks)
bfs.write_submission(sub, 'part1_forecast/submission.csv')          # Codabench scores CSV, not parquet
import zipfile
with zipfile.ZipFile('part1_forecast/submission.zip', 'w', zipfile.ZIP_DEFLATED) as _z:
    _z.write('part1_forecast/submission.csv', 'submission.csv')     # csv at the archive ROOT
print('submission rows:', len(sub), '-> part1_forecast/submission.csv (+ submission.zip to upload)')

## 3. Submit

Upload `part1_forecast/submission.zip` to Codabench.

In [ ]:
print(sub.head(3).to_string())
print('rows', len(sub), '| dims', sorted(sub.horizon.unique()),
      '| q50 mean %.2f m/s' % sub.q50.mean())